In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import ChineseCLIPModel, ChineseCLIPProcessor, CLIPModel, CLIPProcessor
import pandas as pd
from PIL import Image
import numpy as np
import os
from tqdm.auto import tqdm
import pickle
import warnings
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

class PROIInferenceConfig:
    def __init__(self):

        # Dataset version toggle: 'weibo' or 'twitter'.
        # Automatically aligns corresponding hyperparameters, CLIP versions, and I/O paths.
        self.dataset_type = 'weibo'
        
        
        if self.dataset_type == 'weibo':
            # Data loading mode configuration: 'pkl' for serialized data files, 'xlsx' for original spreadsheets.
            self.data_mode = 'pkl'
            self.test_pkl_path = '../autodl-tmp/data/wb_test_3img.pkl'
            self.model_path = '../autodl-tmp/best_model/PROI_wb.pth'
    
            # self.data_mode = 'xlsx'
            # self.test_excel_path = 'weibo_test.xlsx'
            # self.test_img_path = 'wb_img_sum'

            self.result_save_path = 'inference_results_weibo.csv'
            self.clip_model_name = '../model/chinese-clip-vit-base-patch16'
            self.text_max_length = 160
            self.image_size = 224
            self.max_img_num = 3
            
            self.gamma_scale = 30.0
            self.alpha_text = 0.3
            self.beta_smooth_t = 0.9
            self.beta_smooth_i = 0.9
            

            
        elif self.dataset_type == 'twitter':

            self.data_mode = 'pkl'
            self.test_pkl_path = '../autodl-tmp/data/tw_test_3img.pkl'
            self.model_path = '../autodl-tmp/best_model/PROI_tw.pth'
            
            # self.data_mode = 'xlsx'
            # self.test_excel_path = 'tw15_test.xlsx'
            # self.test_img_path = 'tw15_image_224'
            
            self.result_save_path = 'inference_results_twitter.csv'
            self.clip_model_name = '../model/clip-vit-base-patch16'
            self.text_max_length = 77
            self.image_size = 224
            self.max_img_num = 1
            
            self.gamma_scale = 10.0
            self.alpha_text = 0.5
            self.beta_smooth_t = 0.8
            self.beta_smooth_i = 0.8
            
            
        self.batch_size = 32
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.temperature = 0.07 

def conditional_orthogonal_projection(u, v):
    v_norm_sq = torch.sum(v * v, dim=-1, keepdim=True)
    v_norm_sq = torch.where(v_norm_sq == 0, torch.ones_like(v_norm_sq), v_norm_sq)
    dot_product = torch.sum(u * v, dim=-1, keepdim=True)
    proj_scalar = dot_product / v_norm_sq
    proj_vector = proj_scalar * v
    condition = dot_product > 0
    return torch.where(condition, proj_vector, torch.zeros_like(proj_vector))

def absolute_proj_length(u, v):
    dot_product = torch.sum(u * v, dim=-1, keepdim=True)
    v_norm = torch.norm(v, p=2, dim=-1, keepdim=True)
    v_norm = torch.where(v_norm < 1e-9, torch.ones_like(v_norm), v_norm)
    scalar_proj = dot_product / v_norm
    return torch.abs(scalar_proj).squeeze(-1)

class SupConLoss(nn.Module):
    def __init__(self, temperature=0.07, contrast_mode='all', base_temperature=0.07):
        super(SupConLoss, self).__init__()
        self.temperature = temperature
        self.contrast_mode = contrast_mode
        self.base_temperature = base_temperature

    def forward(self, features, labels=None, mask=None):
        device = features.device
        if len(features.shape) < 3: raise ValueError('`features` needs to be [bsz, n_views, ...]')
        if len(features.shape) > 3: features = features.view(features.shape[0], features.shape[1], -1)
        batch_size = features.shape[0]
        if labels is not None and mask is not None: raise ValueError('Cannot define both `labels` and `mask`')
        elif labels is None and mask is None: mask = torch.eye(batch_size, dtype=torch.float32).to(device)
        elif labels is not None:
            labels = labels.contiguous().view(-1, 1)
            if labels.shape[0] != batch_size: raise ValueError('Num of labels does not match num of features')
            mask = torch.eq(labels, labels.T).float().to(device)
        else: mask = mask.float().to(device)
        contrast_count = features.shape[1]
        contrast_feature = torch.cat(torch.unbind(features, dim=1), dim=0)
        if self.contrast_mode == 'one': 
            anchor_feature = features[:, 0]
            anchor_count = 1
        elif self.contrast_mode == 'all': 
            anchor_feature = contrast_feature
            anchor_count = contrast_count
        else: raise ValueError('Unknown mode: {}'.format(self.contrast_mode))
        anchor_dot_contrast = torch.div(torch.matmul(anchor_feature, contrast_feature.T), self.temperature)
        logits_max, _ = torch.max(anchor_dot_contrast, dim=1, keepdim=True)
        logits = anchor_dot_contrast - logits_max.detach()
        mask = mask.repeat(anchor_count, contrast_count)
        logits_mask = torch.scatter(torch.ones_like(mask), 1, torch.arange(batch_size * anchor_count).view(-1, 1).to(device), 0)
        mask = mask * logits_mask
        exp_logits = torch.exp(logits) * logits_mask
        log_prob = logits - torch.log(exp_logits.sum(1, keepdim=True) + 1e-9)
        mean_log_prob_pos = (mask * log_prob).sum(1) / (mask.sum(1) + 1e-9)
        loss = - (self.temperature / self.base_temperature) * mean_log_prob_pos
        loss = loss.view(anchor_count, batch_size).mean()
        return loss

class MultiModalEmbedder(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        if config.dataset_type == 'weibo':
            self.clip_model = ChineseCLIPModel.from_pretrained(config.clip_model_name)
            self.processor = ChineseCLIPProcessor.from_pretrained(config.clip_model_name)
        else:
            self.clip_model = CLIPModel.from_pretrained(config.clip_model_name)
            self.processor = CLIPProcessor.from_pretrained(config.clip_model_name)

    def forward(self, text, news_texts, images):
        device = self.config.device
        text_inputs = self.processor(text=text, return_tensors="pt", padding=True, truncation=True, max_length=self.config.text_max_length).to(device)
        T_cls = self.clip_model.get_text_features(**text_inputs)
        
        flat_news_texts = [n for sublist in news_texts for n in sublist]
        news_inputs = self.processor(text=flat_news_texts, return_tensors="pt", padding=True, truncation=True, max_length=self.config.text_max_length).to(device)
        N_cls_flat = self.clip_model.get_text_features(**news_inputs)
        N_cls = N_cls_flat.view(-1, 4, T_cls.shape[-1])
        
        flat_images = [img for sublist in images for img in sublist]
        image_inputs = self.processor(images=flat_images, return_tensors="pt", padding=True).to(device)
        I_cls = self.clip_model.get_image_features(**image_inputs).view(-1, self.config.max_img_num, T_cls.shape[-1])
        return T_cls, N_cls, I_cls

class DisentanglementAndDistillationModule(nn.Module):
    def __init__(self, config, input_dim=512):
        super().__init__()
        self.config = config
        self.input_dim = input_dim
        self.sup_con_loss = SupConLoss(temperature=config.temperature)
        self.event_direction_extractor = nn.Sequential(nn.Linear(input_dim, input_dim), nn.Sigmoid())

    def forward(self, T_cls, N_cls, I_cls, i_mask):
        device = self.config.device
        T_unit = F.normalize(T_cls, p=2, dim=-1)
        I_unit = F.normalize(I_cls, p=2, dim=-1)
        
        T_norm_scaled = T_unit * self.config.gamma_scale
        I_norm_scaled = I_unit * self.config.gamma_scale
        
        signal_text = self.event_direction_extractor(T_norm_scaled)
        interaction_scaled = (1 - self.config.alpha_text) * I_norm_scaled + self.config.alpha_text * T_norm_scaled.unsqueeze(1)
        signal_image = self.event_direction_extractor(interaction_scaled)

        student_te = T_norm_scaled * signal_text 
        i_mask_expanded = i_mask.unsqueeze(-1)
        student_ie = I_norm_scaled * signal_image * i_mask_expanded
        summed_student_ie = torch.sum(student_ie, dim=1)

        teacher_n = F.normalize(N_cls, p=2, dim=-1)
        student_n_raw = student_te + summed_student_ie
        student_n = F.normalize(student_n_raw, p=2, dim=-1)
        
        features_joint = torch.cat([student_n.unsqueeze(1), teacher_n], dim=1)
        loss_contrastive = self.sup_con_loss(features_joint)
        loss_saturation = signal_text.mean() + signal_image.mean()

        l1 = absolute_proj_length(student_te, student_n_raw)
        l2 = absolute_proj_length(summed_student_ie, student_n_raw)
        c = l1 / (l1 + l2 + 1e-9)

        raw_IE = conditional_orthogonal_projection(I_norm_scaled, student_ie)
        raw_IB = I_norm_scaled - raw_IE
        IE = raw_IE * i_mask_expanded 
        IB = raw_IB * i_mask_expanded 
        TE = conditional_orthogonal_projection(T_norm_scaled, student_te) 
        TS = T_norm_scaled - TE 
        loss_completeness = torch.tensor(0.0, device=device) 
        
        return TS, TE, loss_contrastive, loss_completeness, IE, IB, I_cls, c, loss_saturation, T_norm_scaled, I_norm_scaled

class FeatureInteractionModule(nn.Module):
    def __init__(self, config, input_dim=512):
        super().__init__()
    def forward(self, TE, TS, IE_pool, IB_pool):
        return TE * IB_pool, TS * IE_pool, TE * TS, IE_pool * IB_pool

class GeometryBasedDynamicWeighting(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config

    def forward(self, T_cls, TS, TE, IE, IB, I_cls, i_mask, c):
        sim_ts_tcls = F.cosine_similarity(TS, T_cls, dim=-1)
        sim_te_tcls = F.cosine_similarity(TE, T_cls, dim=-1)
        raw_k1 = sim_ts_tcls / (sim_ts_tcls + sim_te_tcls + 1e-9)
        K1 = self.config.beta_smooth_t * raw_k1 + (1 - self.config.beta_smooth_t) * 0.5

        sim_ib_icls = F.cosine_similarity(IB, I_cls, dim=-1) 
        sim_ie_icls = F.cosine_similarity(IE, I_cls, dim=-1) 
        raw_K2_per_image = sim_ib_icls / (sim_ib_icls + sim_ie_icls + 1e-9)
        smoothed_K2_per_image = self.config.beta_smooth_i * raw_K2_per_image + (1 - self.config.beta_smooth_i) * 0.5

        norm_ie = torch.norm(IE, p=2, dim=-1) 
        img_weights = norm_ie * i_mask 
        weights_sum = img_weights.sum(dim=1, keepdim=True) + 1e-9
        normalized_weights = img_weights / weights_sum 
        weights_expanded = normalized_weights.unsqueeze(-1)
        IE_pool = (IE * weights_expanded).sum(dim=1)
        IB_pool = (IB * weights_expanded).sum(dim=1)
        K2 = (smoothed_K2_per_image * normalized_weights).sum(dim=1)

        k31 = c / (1 - K1 + 1e-9)
        k32 = (1 - c) / (1 - K2 + 1e-9)
        raw_K3 = k31 / (k31 + k32 + 1e-9)
        K3 = self.config.beta_smooth_t * raw_K3 + (1 - self.config.beta_smooth_t) * 0.5
        return K1, K2, K3, IE_pool, IB_pool

class FeatureAggregationAndOutput(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.classifier = nn.Sequential(nn.Linear(512, 2))
        self.eps = 1e-9

    def forward(self, TE, TS, IE_pool, IB_pool, K1, K2, K3, Gx_tuple):
        K1, K2, K3 = K1.unsqueeze(1), K2.unsqueeze(1), K3.unsqueeze(1)
        A = K3 * (1 - K1)
        B = (1 - K3) * (1 - K2)
        K4 = A / (A + B + self.eps)
        one_minus_K4 = 1.0 - K4
        TE_IB, TS_IE, TE_TS, IE_IB = Gx_tuple
        Z2_raw = one_minus_K4 * TS_IE + K4 * TE_TS  
        Z3_raw = K4 * TE_IB + one_minus_K4 * IE_IB  
        Z1 = K3 * (1 - K1) * TE + (1 - K3) * (1 - K2) * IE_pool
        Z = Z1 + K3 * K1 * Z2_raw + (1 - K3) * K2 * Z3_raw
        return self.classifier(Z)

class PROI(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.embedder = MultiModalEmbedder(config)
        self.disentangler = DisentanglementAndDistillationModule(config)
        self.interactor = FeatureInteractionModule(config)
        self.weigher = GeometryBasedDynamicWeighting(config)
        self.aggregator = FeatureAggregationAndOutput(config)

    def forward(self, batch, is_train=False):
        i_mask = batch['i_mask'].to(self.config.device)
        has_news = not (not is_train and all(all(s == "" for s in nt) for nt in batch['news_texts']))
        
        if has_news: 
            T_cls, N_cls, I_cls = self.embedder(batch['text'], batch['news_texts'], batch['images'])
        else:
            T_cls, _, I_cls = self.embedder(batch['text'], [["", "", "", ""]] * len(batch['text']), batch['images'])
            N_cls = torch.zeros(T_cls.shape[0], 4, T_cls.shape[1]).to(self.config.device)
        
        TS, TE, loss_contrastive, loss_completeness, IE, IB, I_cls, c, loss_saturation, T_norm_scaled, I_norm_scaled = self.disentangler(T_cls, N_cls, I_cls, i_mask)
        
        if not has_news: loss_contrastive = torch.tensor(0.0).to(self.config.device)
        K1, K2, K3, IE_pool, IB_pool = self.weigher(T_norm_scaled, TS, TE, IE, IB, I_norm_scaled, i_mask, c)
        Gx_tuple = self.interactor(TE, TS, IE_pool, IB_pool)
        logits = self.aggregator(TE, TS, IE_pool, IB_pool, K1, K2, K3, Gx_tuple)
        
        return logits, loss_contrastive, loss_completeness, loss_saturation

class NewsDataset(Dataset):
    def __init__(self, config, excel_path, img_folder, is_train=False):
        self.config = config
        self.df = pd.read_excel(excel_path)
        self.img_folder = img_folder
        self.is_train = is_train
        self.news_cols = ['news_g1', 'news_g2', 'news_m', 'news_q']
        if not self.is_train:
            for col in self.news_cols:
                if col not in self.df.columns: self.df[col] = ""

    def __len__(self): return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        text = str(row['text']) if pd.notna(row['text']) else ""
        news_texts = [str(row[col]) if pd.notna(row[col]) else "" for col in self.news_cols]
        img_names = str(row['img_name']).split(',') if pd.notna(row['img_name']) else []
        images, valid_img_count = [], 0
        for i in range(self.config.max_img_num):
            if i < len(img_names) and img_names[i]:
                try:
                    images.append(Image.open(os.path.join(self.img_folder, img_names[i])).convert("RGB"))
                    valid_img_count += 1
                except FileNotFoundError: images.append(Image.new('RGB', (self.config.image_size, self.config.image_size)))
            else: images.append(Image.new('RGB', (self.config.image_size, self.config.image_size)))
        i_mask = torch.zeros(self.config.max_img_num)
        i_mask[:valid_img_count] = 1
        label = int(row['label']) if 'label' in self.df.columns else 0 
        return {'text': text, 'news_texts': news_texts, 'images': images, 'i_mask': i_mask, 'label': torch.tensor(label, dtype=torch.long)}

class PKLDataset(Dataset):
    def __init__(self, config, pkl_path, is_train=False):
        self.config = config
        self.is_train = is_train
        print(f"Loading inference dataset: {pkl_path} ...")
        with open(pkl_path, 'rb') as f:
            self.data = pickle.load(f)
            
    def __len__(self): return len(self.data)
    
    def __getitem__(self, idx):
        item = self.data[idx]
        text = item['text']
        news_texts = item['news_texts'] if 'news_texts' in item else [["", "", "", ""]]
        label = item['label'] if 'label' in item else 0
        raw_images = item['images']
        target_num = self.config.max_img_num
        current_images = raw_images[:target_num]
        final_images, valid_img_count = [], 0
        for img in current_images:
            final_images.append(img)
            valid_img_count += 1
        pad_num = target_num - valid_img_count
        for _ in range(pad_num):
            final_images.append(Image.new('RGB', (self.config.image_size, self.config.image_size)))
        i_mask = torch.zeros(target_num)
        i_mask[:valid_img_count] = 1
        return {'text': text, 'news_texts': news_texts, 'images': final_images, 'i_mask': i_mask, 'label': torch.tensor(label, dtype=torch.long)}

def collate_fn(batch):
    return {
        'text': [item['text'] for item in batch], 
        'news_texts': [item['news_texts'] for item in batch], 
        'images': [item['images'] for item in batch], 
        'i_mask': torch.stack([item['i_mask'] for item in batch]), 
        'label': torch.stack([item['label'] for item in batch])
    }

def inference(model, dataloader, config):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    print("Starting Inference...")
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Inferencing"):
            labels = batch['label'].to(config.device)
            logits, _, _, _ = model(batch, is_train=False)
            probs = torch.softmax(logits, dim=1)
            preds = torch.argmax(logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
    return np.array(all_labels), np.array(all_preds), np.array(all_probs)

def main():
    config = PROIInferenceConfig()
    
    if config.data_mode == 'pkl':
        test_dataset = PKLDataset(config, config.test_pkl_path, is_train=False)
    else:
        test_dataset = NewsDataset(config, config.test_excel_path, config.test_img_path, is_train=False)
        
    test_loader = DataLoader(test_dataset, batch_size=config.batch_size, shuffle=False, collate_fn=collate_fn)

    print(f"Initializing PROI model for [{config.dataset_type.upper()}] dataset on {config.device}...")
    model = PROI(config).to(config.device)
    
    if os.path.exists(config.model_path):
        print(f"Loading weights from {config.model_path}...")
        checkpoint = torch.load(config.model_path, map_location=config.device)
        model.load_state_dict(checkpoint)
        print("Model loaded successfully.")
    else:
        print(f"Error: Model file not found at {config.model_path}")
        return

    labels, preds, probs = inference(model, test_loader, config)
    
    acc = accuracy_score(labels, preds)
    precision = precision_score(labels, preds, average='macro', zero_division=0)
    recall = recall_score(labels, preds, average='macro', zero_division=0)
    f1 = f1_score(labels, preds, average='macro', zero_division=0)
    
    print("\n" + "="*50)
    print(f"Inference Results ({config.dataset_type.upper()}):\nAccuracy:  {acc:.4f}\nPrecision: {precision:.4f}\nRecall:    {recall:.4f}\nF1 Score:  {f1:.4f}")
    print("="*50 + "\nDetailed Classification Report:")
    print(classification_report(labels, preds, target_names=['Real', 'Fake'], zero_division=0, digits=4))
    
    results_df = pd.DataFrame({'Label': labels, 'Prediction': preds, 'Prob_Real': probs[:, 0], 'Prob_Fake': probs[:, 1]})
    results_df.to_csv(config.result_save_path, index=False)
    print(f"Detailed results saved to {config.result_save_path}")

if __name__ == '__main__':
    main()


libgomp: Invalid value for environment variable OMP_NUM_THREADS

libgomp: Invalid value for environment variable OMP_NUM_THREADS


Loading inference dataset: ../autodl-tmp/data/wb_test_3img.pkl ...
Initializing PROI model for [WEIBO] dataset on cuda...


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Loading weights from ../autodl-tmp/best_model/PROI_wb.pth...
Model loaded successfully.
Starting Inference...


Inferencing:   0%|          | 0/62 [00:00<?, ?it/s]


Inference Results (WEIBO):
Accuracy:  0.9515
Precision: 0.9516
Recall:    0.9515
F1 Score:  0.9515
Detailed Classification Report:
              precision    recall  f1-score   support

        Real     0.9592    0.9438    0.9514       996
        Fake     0.9439    0.9593    0.9516       983

    accuracy                         0.9515      1979
   macro avg     0.9516    0.9515    0.9515      1979
weighted avg     0.9516    0.9515    0.9515      1979

Detailed results saved to inference_results_weibo.csv
